# C2.6 · Building the research harness

**Function C — Offensive Security & Research → The Security Researcher**  ·  *AI for Security*

Builds on **[C2.5 · Supply-chain research](https://spbreed.github.io/cyber-commons/lessons/C2.5.html)**.

| | |
|---|---|
| Open-source tooling | Inspect, Cyber Commons eval harness |
| Open-weight models | Llama 3.3, GLM-4.6, Kimi K2 |

> **Runs anywhere.** Every line of code is in this notebook — nothing to install, nothing to clone, no API key, no network. Standard library only, so it works on a Kaggle kernel with the internet switched off.

## 1 · The concept


The difference between a person who finds things and a capability that keeps
finding them is a harness: a suite, a target adapter, and recorded rates that
are comparable across runs.

Three properties make it a harness rather than a script:

1. **The suite is data, not code.** Adding a case must not require editing the
   runner.
2. **The target is an adapter.** Pointing it at a new build, a new model or a
   competitor's product should be one function.
3. **Results are comparable.** Same seed, same n, same scoring — so a delta
   means something.

The failure mode to avoid is a harness that only ever produces a number going
down, because the suite is only ever extended with cases the current build
already passes.

## 2 · Demo — suite as data, target as adapter

In [ ]:
import random
from dataclasses import dataclass

@dataclass(frozen=True)
class Case:
    cid: str; surface: str; payload: str; landing_rate: float

SUITE = [
 Case("INJ-01", "injection", "direct override", 0.05),
 Case("INJ-02", "injection", "context reframe", 0.35),
 Case("INJ-03", "injection", "task nesting",    0.62),
 Case("INJ-04", "injection", "authority claim", 0.70),
 Case("IDN-01", "identity",  "scope widening",  0.00),
 Case("IDN-02", "identity",  "impersonation",   0.95),
 Case("CNT-01", "containment", "metadata service", 0.00),
 Case("CNT-02", "containment", "path traversal",   0.10),
]

def trial(p, n, seed):
    rng = random.Random(seed)
    hits = sum(rng.random() < p for _ in range(n))
    rate = hits / n
    half = 1.96 * ((rate * (1 - rate) / n) ** 0.5)
    return {"rate": round(rate, 3),
            "ci95": (round(max(rate-half, 0), 3), round(min(rate+half, 1), 3))}

def run_suite(target, suite=SUITE, n=400, seed=17):
    return {c.cid: {**trial(target(c), n, seed + i), "surface": c.surface}
            for i, c in enumerate(suite)}

def target_baseline(case):        return case.landing_rate
def target_with_provenance(case):
    return 0.02 if case.surface == "injection" else case.landing_rate

base = run_suite(target_baseline)
print(f"{'case':8s}{'surface':13s}{'rate':>7}{'ci95':>18}")
print("-" * 48)
for cid, r in base.items():
    print(f"{cid:8s}{r['surface']:13s}{r['rate']:>7.3f}{str(r['ci95']):>18}")

## 3 · The comparison a harness exists to produce

In [ ]:
after = run_suite(target_with_provenance)

print(f"{'case':8s}{'before':>9}{'after':>9}{'delta':>9}  demonstrated?")
print("-" * 56)
for cid in base:
    b, a = base[cid], after[cid]
    overlap = a["ci95"][1] >= b["ci95"][0]
    print(f"{cid:8s}{b['rate']:>9.3f}{a['rate']:>9.3f}{a['rate']-b['rate']:>+9.3f}"
          f"  {'no — intervals overlap' if overlap else 'yes'}")

def surface_asr(results):
    out = {}
    for cid, r in results.items():
        d = out.setdefault(r["surface"], [])
        d.append(r["rate"])
    return {k: round(sum(v)/len(v), 3) for k, v in out.items()}
print(f"\nbefore by surface: {surface_asr(base)}")
print(f"after  by surface: {surface_asr(after)}")

## 4 · Where it breaks — a suite that only ever grows easier

The metric that makes a research programme look productive while measuring nothing: add cases the current build already passes, and the aggregate ASR falls every quarter.

In [ ]:
EASY = [Case(f"EASY-{i:02d}", "injection", "already blocked", 0.00)
        for i in range(1, 13)]

for label, suite in (("original suite", SUITE),
                     ("suite + 12 easy cases", SUITE + EASY)):
    r = run_suite(target_baseline, suite)
    asr = sum(x["rate"] for x in r.values()) / len(r)
    print(f"{label:26s} cases={len(suite):>3}  aggregate ASR {asr:.3f}")
print("\nThe build did not change. The number improved by 60%.")
print("Report per-surface and per-case, and state when cases were added.")

In [ ]:
# Verify: guard against suite dilution.
def suite_health(suite, results):
    unblocked = [c for c in suite if results[c.cid]["rate"] > 0.05]
    return {"cases": len(suite),
            "still_landing": len(unblocked),
            "trivially_blocked": len(suite) - len(unblocked),
            "dilution_ratio": round((len(suite)-len(unblocked))/len(suite), 2),
            "healthy": (len(suite)-len(unblocked))/len(suite) < 0.7}

for label, suite in (("original", SUITE), ("diluted", SUITE + EASY)):
    r = run_suite(target_baseline, suite)
    h = suite_health(suite, r)
    print(f"{label:12s}{h}")
assert not suite_health(SUITE + EASY, run_suite(target_baseline, SUITE + EASY))["healthy"]

## What you just proved

The baseline suite reports per-case rates with intervals. Provenance reduces every injection case to about 0.02 with non-overlapping intervals, while identity and containment are unchanged. Adding 12 trivially-blocked cases cuts aggregate ASR by roughly 60% with no change to the build, and the suite-health check flags that suite as diluted.

## Your turn

Check your own security regression suite for dilution: what fraction of its cases have ever failed? If it is under 30%, the aggregate number it produces is mostly measuring how many easy cases you added.

---

**Next → [C2.7 · Benchmark design and critique](https://spbreed.github.io/cyber-commons/lessons/C2.7.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/C2.6.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/C2.6.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*